In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
from urllib3.exceptions import ReadTimeoutError
from urllib.parse import urljoin
from datetime import datetime
import time
import os
import json


    
# 1. MAIN SEARCH PAGE
    

BASE_URL = "https://saudi.tanqeeb.com"

SEARCH_PATH = "https://saudi.tanqeeb.com/jobs/search"

QUERY_STRING = (
    "keywords=&country=54&state=0&category=-1&"
    "workplace=0&search_period=0&lang=all&change_lang=1"
)

SOURCE_NAME = "Tanqeeb"


def format_datetime(dt):
    """
    Format a datetime as e.g. '11/9/2026 4:54:00 PM'
    (no leading zeros on day/month/hour, cross-platform).
    """
    if dt is None or pd.isna(dt):
        return "N/A"

    hour_12 = dt.hour % 12
    if hour_12 == 0:
        hour_12 = 12

    am_pm = "AM" if dt.hour < 12 else "PM"

    return f"{dt.day}/{dt.month}/{dt.year} {hour_12}:{dt.minute:02d}:{dt.second:02d} {am_pm}"

# Standardized output paths
JSON_OUTPUT = "../data/raw/tanqeeb _jobs.json"
CSV_OUTPUT = "../data/raw/tanqeeb _jobs.csv"

# Checkpoint files so a crash never costs you all the work done so far
URLS_CHECKPOINT = "../data/raw/tanqeeb_job_urls_checkpoint.json"
JOBS_CHECKPOINT = "../data/raw/tanqeeb_jobs_checkpoint.json"

# How often (in scraped jobs) to flush progress to disk in step 3
JOBS_SAVE_EVERY = 50

# Final standardized column order
STANDARD_COLUMNS = [
    "job_id",
    "source",
    "job_title",
    "company_name",
    "job_description",
    "job_url",
    "country",
    "city",
    "work_mode",
    "employment_type",
    "career_level",
    "experience_needed",
    "qualification_required",
    "skills",
    "salary",
    "estimated_salary",
    "posted_date",
    "collected_at",
]

# Safety cap in case pagination never signals "end" cleanly.
# ~50,700 jobs / ~28 per page is roughly 1,800+ pages, so raise
# this if you actually intend to scrape the entire site.
MAX_PAGES = 2000

# Stop pagination after this many consecutive driver failures in a row
MAX_CONSECUTIVE_FAILURES = 5


    
# 2. OPEN SEARCH PAGE WITH SELENIUM AND WALK THROUGH PAGES

def build_page_url(page_num):
    """
    Tanqeeb paginates using a path segment, not a query param:
      Page 1: https://saudi.tanqeeb.com/jobs/search?keywords=...
      Page 2: https://saudi.tanqeeb.com/jobs/search/page/2?keywords=...
      Page N: https://saudi.tanqeeb.com/jobs/search/page/N?keywords=...
    """
    if page_num <= 1:
        return f"{SEARCH_PATH}?{QUERY_STRING}"
    return f"{SEARCH_PATH}/page/{page_num}?{QUERY_STRING}"


def get_job_links_from_html(html):
    soup = BeautifulSoup(html, "html.parser")
    return soup.select("h2.search-job-title a.search-job-title-link")


def make_driver():
    """Create a fresh headless Chrome driver with a longer command timeout."""
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    d = webdriver.Chrome(options=options)
    # Default Selenium command timeout is 120s; bump it since some
    # search pages render slowly under headless Chrome after many navigations.
    d.command_executor.set_timeout(300)
    return d


def save_urls_checkpoint(urls, last_completed_page):
    os.makedirs(os.path.dirname(URLS_CHECKPOINT), exist_ok=True)
    with open(URLS_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(
            {"urls": urls, "last_completed_page": last_completed_page},
            f,
            ensure_ascii=False,
            indent=2,
        )


def load_urls_checkpoint():
    if os.path.exists(URLS_CHECKPOINT):
        with open(URLS_CHECKPOINT, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("urls", []), data.get("last_completed_page", 0)
    return [], 0


# Resume from a previous run if a checkpoint exists
job_urls, last_completed_page = load_urls_checkpoint()
page_num = last_completed_page + 1

if job_urls:
    print(f"Resuming pagination from page {page_num}. "
          f"{len(job_urls)} URLs already collected.")

driver = make_driver()
consecutive_failures = 0

try:
    while page_num <= MAX_PAGES:

        page_url = build_page_url(page_num)

        print(f"\nLoading search page {page_num}: {page_url}")

        try:
            driver.get(page_url)
            # Wait for page to load / JS to render results
            time.sleep(5)
            html = driver.page_source
            consecutive_failures = 0
        except (TimeoutException, WebDriverException, ReadTimeoutError) as e:
            consecutive_failures += 1
            print(f"  !! Driver error on page {page_num} "
                  f"(failure {consecutive_failures}/{MAX_CONSECUTIVE_FAILURES}): {e}")

            # Kill and restart the browser process; retry the same page
            try:
                driver.quit()
            except Exception:
                pass
            time.sleep(3)
            driver = make_driver()

            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                print("Too many consecutive driver failures. Stopping pagination "
                      "(progress up to the last completed page is saved).")
                break

            continue  # retry page_num with the fresh driver, don't advance

        job_links = get_job_links_from_html(html)

        print(f"  -> Job links found on this page: {len(job_links)}")

        # Stop condition: no job cards on this page means we've
        # gone past the last page of results.
        if not job_links:
            print("No more job listings found. Stopping pagination.")
            break

        page_had_new_url = False

        for link in job_links:
            href = link.get("href")
            if href:
                full_url = urljoin(BASE_URL, href)
                if full_url not in job_urls:
                    job_urls.append(full_url)
                    page_had_new_url = True

        # Extra safety: if a page returns only links we've already
        # seen (e.g. the site clamped an out-of-range page back to
        # the last page), treat that as the end too.
        if not page_had_new_url:
            print("No new job URLs on this page (likely repeated last page). Stopping.")
            break

        # Persist progress after every successfully processed page
        save_urls_checkpoint(job_urls, page_num)

        page_num += 1

finally:
    driver.quit()


# Remove duplicates while preserving order (belt-and-suspenders,
# since we already dedupe while collecting above)
job_urls = list(dict.fromkeys(job_urls))

# Final save of the URL checkpoint
save_urls_checkpoint(job_urls, page_num - 1)

print("\nUnique job URLs collected across all pages:", len(job_urls))


    
# 3. SCRAPE EACH JOB PAGE

def save_jobs_checkpoint(jobs):
    os.makedirs(os.path.dirname(JOBS_CHECKPOINT), exist_ok=True)
    with open(JOBS_CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(jobs, f, ensure_ascii=False, indent=2)


def load_jobs_checkpoint():
    if os.path.exists(JOBS_CHECKPOINT):
        with open(JOBS_CHECKPOINT, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


# Resume job-scraping progress too: skip URLs already scraped
all_jobs = load_jobs_checkpoint()
already_scraped_urls = {job.get("job_url") for job in all_jobs}

if already_scraped_urls:
    print(f"Resuming job scraping. {len(already_scraped_urls)} jobs already scraped.")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

for i, url in enumerate(job_urls, start=1):

    if url in already_scraped_urls:
        continue

    print(f"\n[{i}/{len(job_urls)}] Scraping:")
    print(url)

    try:
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code != 200:
            print("Failed:", response.status_code)
            continue

        soup = BeautifulSoup(response.text, "html.parser")

  
        # JOB ID
        apply_button = soup.select_one(".apply-btn")
        job_id = apply_button.get("data-job-id") if apply_button else None

  
        # JOB TITLE
        title_element = soup.select_one("h3.job-title-text")
        job_title = title_element.get_text(" ", strip=True) if title_element else None

  
        # COMPANY NAME
        company_element = soup.select_one(
            "a.job-meta-company, .job-company-name, .company-name, a.company-name-link"
        )
        company_name = (
            company_element.get_text(" ", strip=True)
            if company_element else None
        )

  
        # WORK MODE + JOB TYPE
        tags = soup.select(".job-tags .job-tag")
        work_mode = None
        job_type = None

        for tag in tags:
            value = tag.get_text(" ", strip=True)
            value_lower = value.lower()

            if value_lower in ["on-site", "remote", "hybrid"]:
                work_mode = value
            elif value_lower in [
                "full time", "part time", "contract",
                "internship", "temporary", "freelance"
            ]:
                job_type = value

  
        # LOCATION
        location_element = soup.select_one(".job-meta-item span")
        country = None
        city = None

        if location_element:
            location_text = location_element.get_text(" ", strip=True)
            parts = [x.strip() for x in location_text.split(",")]

            if len(parts) >= 1:
                country = parts[0]
            if len(parts) >= 2:
                city = parts[1]

  
        # POSTED DATE
        date_element = soup.select_one(".job-date")
        raw_posted_date = date_element.get("data-datetime") if date_element else None

        parsed_posted_date = pd.to_datetime(raw_posted_date, errors="coerce") if raw_posted_date else None
        posted_date = format_datetime(parsed_posted_date) if parsed_posted_date is not None else "N/A"

  
        # JOB DETAILS

        job_details = {}
        meta_rows = soup.select(".meta-data .meta")

        for row in meta_rows:
            spans = row.find_all("span")
            if len(spans) >= 2:
                label = spans[0].get_text(" ", strip=True)
                value = spans[1].get_text(" ", strip=True)
                job_details[label] = value

        experience = job_details.get("Experience")
        employment = job_details.get("Employment")
        gender = job_details.get("Gender")
        education = job_details.get("Education")
        industry = job_details.get("Industry")
        career_level = job_details.get("Career Level")
        salary = job_details.get("Salary")

  
        # SKILLS
        skill_elements = soup.select(".job-skills .skill, .skills-list .skill-tag")
        skills_list = [
            s.get_text(" ", strip=True)
            for s in skill_elements
            if s.get_text(strip=True)
        ]
        skills = "; ".join(skills_list) if skills_list else None

  
        # JOB DESCRIPTION
        description_element = soup.select_one("#jobDescriptionBody")
        description = (
            description_element.get_text(" ", strip=True)
            if description_element else None
        )

  
        # SAVE JOB (standardized schema)
        job = {
            "job_id": job_id if job_id else "N/A",
            "source": SOURCE_NAME,
            "job_title": job_title if job_title else "N/A",
            "company_name": company_name if company_name else "N/A",
            "job_description": description if description else "N/A",
            "job_url": url if url else "N/A",
            "country": country if country else "N/A",
            "city": city if city else "N/A",
            "work_mode": work_mode if work_mode else "N/A",
            "employment_type": employment if employment else (job_type if job_type else "N/A"),
            "career_level": career_level if career_level else "N/A",
            "experience_needed": experience if experience else "N/A",
            "qualification_required": education if education else "N/A",
            "skills": skills if skills else "N/A",
            "salary": salary if salary else "N/A",
            "estimated_salary": "",
            "posted_date": posted_date if posted_date else "N/A",
            "collected_at": format_datetime(datetime.now()),
        }

        all_jobs.append(job)
        already_scraped_urls.add(url)

        print("✓ Scraped:", job_title)

        # Periodically flush progress so a crash doesn't lose everything
        if len(all_jobs) % JOBS_SAVE_EVERY == 0:
            save_jobs_checkpoint(all_jobs)
            print(f"  (checkpoint saved: {len(all_jobs)} jobs so far)")

    except Exception as e:
        print("Error:", e)

    # Wait between requests
    time.sleep(1)

# Final checkpoint save after the loop completes
save_jobs_checkpoint(all_jobs)

    
# 4. CREATE DATAFRAME (standardized column order) 

df = pd.DataFrame(all_jobs)

# Ensure every standardized column exists, even if a field was
# never populated for any job (keeps output schema consistent).
for col in STANDARD_COLUMNS:
    if col not in df.columns:
        df[col] = "N/A"

df = df[STANDARD_COLUMNS]

    
# 5. SAVE CSV + JSON

os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)
os.makedirs(os.path.dirname(JSON_OUTPUT), exist_ok=True)

df.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")

with open(JSON_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(all_jobs, f, ensure_ascii=False, indent=2)

# 6. RESULTS

print("\n" + "=" * 60)
print("Total URLs:", len(job_urls))
print("Successfully scraped:", len(df))
print("CSV saved as:", CSV_OUTPUT)
print("JSON saved as:", JSON_OUTPUT)
print("=" * 60)

df.head()


Loading search page 1: https://saudi.tanqeeb.com/jobs/search?keywords=&country=54&state=0&category=-1&workplace=0&search_period=0&lang=all&change_lang=1
  -> Job links found on this page: 27

Loading search page 2: https://saudi.tanqeeb.com/jobs/search/page/2?keywords=&country=54&state=0&category=-1&workplace=0&search_period=0&lang=all&change_lang=1
  -> Job links found on this page: 20

Loading search page 3: https://saudi.tanqeeb.com/jobs/search/page/3?keywords=&country=54&state=0&category=-1&workplace=0&search_period=0&lang=all&change_lang=1
  -> Job links found on this page: 20

Loading search page 4: https://saudi.tanqeeb.com/jobs/search/page/4?keywords=&country=54&state=0&category=-1&workplace=0&search_period=0&lang=all&change_lang=1
  -> Job links found on this page: 20

Loading search page 5: https://saudi.tanqeeb.com/jobs/search/page/5?keywords=&country=54&state=0&category=-1&workplace=0&search_period=0&lang=all&change_lang=1
  -> Job links found on this page: 20

Loading sea

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=63865): Read timed out. (read timeout=120)

1 page = 10 ÷ 60 = 0.167 minutes
500 pages = 500 × 0.167 = 83.3 minutes =1 hour and 23 minutes